# Stima di un sistema di domanda residenziale di energia con regressione apparentemente non correlata

## Sintesi esecutiva

Una utility regionale stima un **sistema di domanda residenziale di energia** a due equazioni — le quote di bilancio dell'elettricità e del gas naturale in funzione del prezzo proprio, del prezzo incrociato e del reddito familiare — usando **PROC SYSLIN** con il metodo della regressione apparentemente non correlata (SUR). Le due equazioni delle quote condividono un budget familiare comune, quindi i loro errori sono correlati tra le equazioni; la SUR stima le equazioni congiuntamente per sfruttare tale correlazione, recuperando effetti di prezzo proprio e incrociato coerenti nei segni e fornendo la covarianza tra le equazioni e i test sulle restrizioni di cui un analista della domanda ha bisogno.

## Fonti dei dati

| Dataset | Righe | Granularità | Variabili chiave | Descrizione |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | una riga per osservazione mensile di mercato | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Panel sintetico mensile del mercato residenziale dell'energia. `lp_elec`/`lp_gas` sono i log dei prezzi reali dell'elettricità e del gas naturale; `lincome` è il log del reddito familiare reale; `w_elec`/`w_gas` sono le quote di bilancio di spesa per elettricità e gas naturale, generate da una struttura di domanda nota di tipo AIDS più rumore correlato tra le equazioni. |

# Stima di un sistema di domanda residenziale di energia con regressione apparentemente non correlata

Una utility regionale di gas ed elettricità vuole capire come le famiglie riallocano il loro budget energetico tra **elettricità** e **gas naturale** al variare dei prezzi relativi e del reddito. Il quadro naturale è un **sistema di domanda**: un insieme di equazioni delle quote di bilancio stimate congiuntamente.

Due caratteristiche rendono la stima congiunta lo strumento giusto:

- Le equazioni delle quote attingono a un budget familiare comune, quindi i loro errori sono **correlati tra le equazioni** — quando una famiglia spende di più per l'elettricità spende di meno per il gas. Stimare le equazioni insieme con la **regressione apparentemente non correlata (SUR)** sfrutta tale correlazione per ottenere efficienza.
- La teoria economica impone **restrizioni lineari** (somma, omogeneità, simmetria) che uno stimatore di sistema può imporre direttamente.

`PROC SYSLIN` con l'opzione `SUR` è pensato esattamente per questa situazione. Adatta ciascuna equazione delle quote, stima la covarianza degli errori tra le equazioni e riadatta il sistema con quella covarianza per fornire elasticità efficienti e coerenti con la teoria — insieme alla matrice di covarianza tra i modelli e ai test congiunti sulle restrizioni.

In questo notebook:

1. Generiamo un panel sintetico mensile del mercato dell'energia da una struttura nota delle quote.
2. Adattiamo un sistema di quote a due equazioni con la SUR.
3. Confrontiamo l'adattamento SUR congiunto con la stima OLS equazione per equazione.
4. Imponiamo una restrizione di omogeneità e leggiamo il relativo test F congiunto.
5. Estraiamo la tabella dei coefficienti per la reportistica delle elasticità.

## Passo 1 — Generare un panel sintetico del mercato dell'energia

Simuliamo 60 osservazioni mensili di un piccolo **sistema di domanda di energia a due beni** (elettricità e gas naturale). Il processo generatore dei dati è un modello di quota di bilancio linearizzato, in stile AIDS:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

Costruiamo deliberatamente una **correlazione tra le equazioni**: quando le famiglie spendono di più per l'elettricità spendono di meno per il gas, quindi `e1` ed `e2` sono correlati negativamente. Un fattore di prezzo comune del mercato dell'energia guida inoltre entrambi i log-prezzi insieme. Sono queste le caratteristiche che ripagano la stima congiunta SUR rispetto all'adattamento di ciascuna equazione da sola.

In [1]:
DATI energy;
   CHIAMARE streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   FARE month = 1 FINO_A 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      USCITA;
   FINE;

   MANTENERE month lp_elec lp_gas lincome w_elec w_gas;
ESEGUIRE;

PROCEDURA MEDIE DATI=energy n mean std MIN MAX maxdec=3;
   VARIABILE w_elec w_gas lp_elec lp_gas lincome;
ESEGUIRE;

                                                  The MEANS Procedure

 Variable        N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 W_ELEC         60          0.440       0.011       0.417       0.464
 W_GAS          60          0.228       0.014       0.198       0.256
 LP_ELEC        60          4.617       0.142       4.354       4.911
 LP_GAS         60          4.211       0.162       3.818       4.566
 LINCOME        60         10.916       0.088      10.723      11.174
 --------------------------------------------------------------------



NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Passo 2 — Stima SUR di base del sistema di domanda

Stimiamo entrambe le equazioni delle quote congiuntamente. L'opzione `SUR` indica a `PROC SYSLIN` di stimare la covarianza degli errori tra le equazioni e di usarla per un adattamento GLS fattibile ed efficiente. Due istruzioni `MODEL` etichettate (`elec` e `gas`) definiscono il sistema; ciascuna regredisce una quota di bilancio sui due log-prezzi e sul log del reddito.

SYSLIN riporta le stime dei parametri di ciascuna equazione e, alla fine, la **Cross-Model Covariance Matrix** — la covarianza stimata degli errori delle due equazioni che la SUR sfrutta.

In [2]:
PROCEDURA syslin DATI=energy ON;
   elec: MODELLO w_elec = lp_elec lp_gas lincome;
   gas:  MODELLO w_gas  = lp_elec lp_gas lincome;
ESEGUIRE;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Passo 3 — Confronto con la stima OLS equazione per equazione

Per vedere cosa ci offre la SUR, riadattiamo le stesse due equazioni con i minimi quadrati ordinari (il metodo predefinito, una equazione alla volta). L'OLS ignora la correlazione degli errori tra le equazioni, quindi è consistente ma meno efficiente della SUR quando tale correlazione è diversa da zero — come lo è qui per costruzione.

Il confronto delle due tabelle dei coefficienti mostra come le stime e i loro errori standard cambiano una volta tenuto conto della struttura del sistema.

In [3]:
PROCEDURA syslin DATI=energy ols;
   elec: MODELLO w_elec = lp_elec lp_gas lincome;
   gas:  MODELLO w_gas  = lp_elec lp_gas lincome;
ESEGUIRE;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Passo 4 — Imporre la teoria economica e verificarla

La teoria della domanda afferma che gli effetti nominali di prezzo/reddito dovrebbero rispettare l'**omogeneità di grado zero**: riscalare tutti i prezzi e il reddito lascia invariata la domanda reale, quindi i coefficienti di prezzo e reddito all'interno di un'equazione sommano a zero. Imponiamo questo sulla quota dell'elettricità con un'istruzione `RESTRICT`.

SYSLIN riadatta il sistema soggetto al vincolo e riporta un test F dei **Restriction Results** su se la restrizione sia coerente con i dati — un test diretto e congiunto dell'ipotesi di omogeneità.

In [4]:
PROCEDURA syslin DATI=energy ON;
   elec: MODELLO w_elec = lp_elec lp_gas lincome;
   gas:  MODELLO w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
ESEGUIRE;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Passo 5 — Catturare le stime per la reportistica delle elasticità

Per un report finale rendiamo persistenti i coefficienti convergiti con `OUTEST=`. Il dataset risultante contiene una riga per equazione con l'intercetta e le stime delle pendenze più le statistiche di adattamento, che alimentano i calcoli delle elasticità di prezzo proprio e incrociato della utility alle quote medie campionarie. Lo stampiamo per registrazione.

In [5]:
PROCEDURA syslin DATI=energy ON outest=demand_est;
   elec: MODELLO w_elec = lp_elec lp_gas lincome;
   gas:  MODELLO w_gas  = lp_elec lp_gas lincome;
ESEGUIRE;

PROCEDURA STAMPARE DATI=demand_est noobs;
   TITOLO "SUR demand-system coefficient estimates";
ESEGUIRE;
TITOLO;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Interpretare i risultati

**Coerenza dei segni.** Le stime SUR recuperano la struttura di sostituzione incorporata nei dati. Nell'equazione della quota dell'elettricità il **coefficiente del log-prezzo proprio è positivo** (`LP_ELEC` = 0.112) mentre il **coefficiente del prezzo incrociato è negativo** (`LP_GAS` = -0.098); nell'equazione della quota del gas lo schema è speculare (proprio `LP_GAS` = 0.114, incrociato `LP_ELEC` = -0.075). Ogni effetto di prezzo proprio e incrociato è statisticamente significativo (ogni `Pr > |t|` inferiore a 0.005), quindi i segni della sostituzione sono identificati con solidità e non sono artefatti di un adattamento rumoroso.

**La SUR sfrutta la correlazione tra le equazioni.** La **Cross-Model Covariance Matrix** mostra un elemento fuori diagonale negativo (-0.000068): le famiglie scambiano la spesa per l'elettricità con quella per il gas, esattamente come costruito. Poiché tale covarianza è diversa da zero, la stima congiunta SUR è più efficiente dell'adattamento di ciascuna equazione da sola — gli errori standard SUR nel Passo 2 sono uniformemente un po' più piccoli degli errori standard OLS equazione per equazione nel Passo 3 (per esempio, l'errore standard di `LP_ELEC` per il gas scende da 0.0264 con OLS a 0.0255 con SUR), mentre le stime puntuali sono invariate. Questa inferenza più stretta è il vantaggio del modellare il sistema anziché due regressioni separate.

**La teoria regge.** Imporre l'**omogeneità di grado zero** sulla quota dell'elettricità (coefficienti di prezzo e reddito che sommano a zero) tramite `RESTRICT` ha prodotto un test F dei Restriction Results pari a 2.14 con `Pr > F` = 0.149. La restrizione **non viene rifiutata**, quindi la previsione di omogeneità della teoria della domanda del consumatore è coerente con questo campione — la utility può portare con fiducia le stime vincolate e coerenti con la teoria nella reportistica.

**Uso operativo.** La tabella `OUTEST=` rende persistente una riga per equazione con l'intercetta e le stime delle pendenze e le statistiche di adattamento. Quei coefficienti si convertono direttamente in elasticità di prezzo proprio e incrociato e in un'elasticità di reddito (spesa) alle quote medie campionarie (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 dal Passo 1), dando alla utility una base difendibile e coerente con la teoria per la previsione della domanda e gli scenari di progettazione tariffaria.